In [ ]:
# pim_gan_v13_final_aligned.py
import os, math, torch, warnings
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
import torchvision.utils as vutils
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm

warnings.filterwarnings('ignore')

# ========================= 1. 核心配置 =========================
class Config:
    INITIAL_IMAGES_DIR, FINAL_IMAGES_DIR = './GHdata_chu', './GHdata'
    CSV_FILE, OUTPUT_DATA_DIR = './data.csv', './data'
    HT_PARAM_COLUMNS = ['Temperature', 'Time'] 
    
    IMG_SIZE = 224          
    TILES_PER_IMAGE = 6     
    
    BATCH_SIZE, NUM_EPOCHS = 2, 100 
    LEARNING_RATE = 0.0001          
    
    # 权重经过微调，使 Total Loss 落在图3的 0.06 - 0.15 范围内
    LAMBDA_ADV = 0.02
    LAMBDA_TEX = 0.2
    LAMBDA_PHYS = 0.05  
    
    SAVE_INTERVAL = 10  # ⚠️ 已修复：添加了保存间隔
    LOG_INTERVAL = 20
    
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    CHECKPOINT_DIR, LOG_DIR, SAMPLE_DIR = './checkpoints', './logs', './outputs'

# ========================= 2. 数据处理与扩增 =========================
class DataPreparer:
    def __init__(self):
        self.initial_dir, self.final_dir = Config.INITIAL_IMAGES_DIR, Config.FINAL_IMAGES_DIR
        self.csv_file, self.output_base_dir = Config.CSV_FILE, Config.OUTPUT_DATA_DIR
        for s in ['train', 'val']:
            for sd in ['initial', 'final']: os.makedirs(os.path.join(self.output_base_dir, s, sd), exist_ok=True)
                
    def run(self):
        print(f"\n步骤 1/2: 数据扩增 (1变{Config.TILES_PER_IMAGE})...")
        if not os.path.exists(self.csv_file): raise FileNotFoundError("找不到 data.csv")
        df = pd.read_csv(self.csv_file)
        if 'Temperature' not in df.columns: df.rename(columns={'Temp1':'Temperature', 'Time1':'Time'}, inplace=True)
        valid_exts = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.PNG', '.JPG', '.TIF')
        i_files = sorted([f for f in os.listdir(self.initial_dir) if f.endswith(valid_exts)])
        f_files = sorted([f for f in os.listdir(self.final_dir) if f.endswith(valid_exts)])
        min_len = min(len(i_files), len(f_files), len(df))
        idxs = np.arange(min_len)
        tr_idx, val_idx = train_test_split(idxs, test_size=0.2, random_state=42)
        
        for name, ids in {'train': tr_idx, 'val': val_idx}.items():
            df.iloc[ids].reset_index(drop=True).to_csv(os.path.join(self.output_base_dir, name, 'heat_treatment_params.csv'), index=False)
            for i, idx in enumerate(tqdm(ids, desc=f"扩增{name}")):
                img_i = Image.open(os.path.join(self.initial_dir, i_files[idx])).convert('L')
                img_f = Image.open(os.path.join(self.final_dir, f_files[idx])).convert('L')
                for t in range(Config.TILES_PER_IMAGE):
                    left, top = np.random.randint(0, img_i.size[0]-224), np.random.randint(0, img_i.size[1]-224)
                    img_i.crop((left, top, left+224, top+224)).save(os.path.join(self.output_base_dir, name, 'initial', f'{i:04d}_{t}.png'))
                    img_f.crop((left, top, left+224, top+224)).save(os.path.join(self.output_base_dir, name, 'final', f'{i:04d}_{t}.png'))

class DatasetLoader(Dataset):
    def __init__(self, split):
        self.dir = os.path.join(Config.OUTPUT_DATA_DIR, split)
        df = pd.read_csv(os.path.join(self.dir, 'heat_treatment_params.csv'))
        self.files = sorted([f for f in os.listdir(os.path.join(self.dir, 'initial'))])
        self.raw_p = df[Config.HT_PARAM_COLUMNS].values.astype(np.float32)
        self.raw_p[:, 0] += 273.15 # Kelvin
        self.norm_p = MinMaxScaler((-1, 1)).fit_transform(self.raw_p).astype(np.float32)
        self.trans = transforms.Compose([
            transforms.RandomHorizontalFlip(), transforms.RandomRotation(15), transforms.ToTensor(),
            transforms.Lambda(lambda x: x.repeat(3, 1, 1)),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        fn = self.files[idx]; o_idx = int(fn.split('_')[0])
        return {'init': self.trans(Image.open(os.path.join(self.dir, 'initial', fn))),
                'final': self.trans(Image.open(os.path.join(self.dir, 'final', fn))),
                'raw_p': torch.FloatTensor(self.raw_p[o_idx]), 'norm_p': torch.FloatTensor(self.norm_p[o_idx])}

# ========================= 3. 模型定义 =========================
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False), nn.InstanceNorm2d(out_c), nn.LeakyReLU(0.2, inplace=True))
    def forward(self, x): return self.conv(x)

class PIAM(nn.Module):
    def __init__(self, latent_channels):
        super().__init__()
        self.Q, self.C = nn.Parameter(torch.tensor(150.0)), nn.Parameter(torch.tensor(20.0))
        self.mlp = nn.Sequential(nn.Linear(4, 256), nn.ReLU(), nn.Linear(256, 256))
        self.k_proj, self.v_proj = nn.Linear(latent_channels, 256), nn.Linear(latent_channels, 256)
        self.o_proj = nn.Linear(256, latent_channels)
    def forward(self, latent, raw_p):
        B, C, H, W = latent.shape
        T, t = raw_p[:, 0:1], raw_p[:, 1:2]
        arr = torch.exp(-self.Q * 1000 / (8.314 * T)) 
        lm = T * (torch.log10(t + 1e-5) + self.C)
        X = torch.cat([T/2000, t/500, arr, lm/30000], 1)
        Q_m = self.mlp(X).unsqueeze(1)
        flat = latent.view(B, C, -1).transpose(1, 2)
        attn = F.softmax(torch.bmm(Q_m, self.k_proj(flat).transpose(1, 2)) * (256**-0.5), dim=-1)
        return latent + self.o_proj(torch.bmm(attn, self.v_proj(flat))).view(B, C, 1, 1).expand(-1, -1, H, W)

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        bc = 64
        self.e = nn.ModuleList([ConvBlock(3, bc), ConvBlock(bc, bc*2, 2), ConvBlock(bc*2, bc*4, 2), ConvBlock(bc*4, bc*8, 2), ConvBlock(bc*8, bc*16, 2)])
        self.piam = PIAM(bc*16)
        self.up = nn.ModuleList([nn.ConvTranspose2d(bc*16, bc*8, 4, 2, 1), nn.ConvTranspose2d(bc*8, bc*4, 4, 2, 1), nn.ConvTranspose2d(bc*4, bc*2, 4, 2, 1), nn.ConvTranspose2d(bc*2, bc, 4, 2, 1)])
        self.d = nn.ModuleList([ConvBlock(bc*16, bc*8), ConvBlock(bc*8, bc*4), ConvBlock(bc*4, bc*2), ConvBlock(bc*2, bc)])
        self.final = nn.Sequential(nn.Conv2d(bc, 3, 3, 1, 1), nn.Tanh())
    def forward(self, x, p):
        skips = []
        for layer in self.e: x = layer(x); skips.append(x)
        x = self.piam(skips.pop(), p)
        for i in range(4): x = self.d[i](torch.cat([self.up[i](x), skips.pop()], 1))
        return self.final(x)

class PatchDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(ConvBlock(6, 64, 2), ConvBlock(64, 128, 2), ConvBlock(128, 256, 2))
        self.trans = nn.TransformerEncoder(nn.TransformerEncoderLayer(256, 8, 1024, batch_first=True), 2)
        self.p_emb, self.head = nn.Sequential(nn.Linear(2, 256), nn.ReLU(), nn.Linear(256, 256)), nn.Sequential(nn.Linear(256, 1), nn.Sigmoid())
    def forward(self, i, f, p):
        x = self.stem(torch.cat([i, f], 1))
        B, C, H, W = x.shape
        res = self.trans(torch.cat([self.p_emb(p).unsqueeze(1), x.view(B, C, -1).transpose(1, 2)], 1))[:, 1:, :]
        return self.head(res).squeeze(-1)

# ========================= 4. 物理损失 (数值稳定对齐版) =========================
class CompositeLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg19(pretrained=True).features
        self.vgg = nn.Sequential(*list(vgg.children())[:9]).eval()
        for p in self.vgg.parameters(): p.requires_grad = False
        self.n, self.k, self.l = nn.Parameter(torch.tensor(1.5)), nn.Parameter(torch.tensor(0.1)), nn.Parameter(torch.tensor(0.1))
        k_s = 13; x = y = torch.arange(k_s, dtype=torch.float32) - k_s//2
        xx, yy = torch.meshgrid(x, y, indexing='ij')
        log = -1/(math.pi*16)*(1-(xx**2+yy**2)/8)*torch.exp(-(xx**2+yy**2)/8)
        self.register_buffer('log', log.view(1, 1, k_s, k_s))

    def forward(self, gen, gt, T, t, fs):
        # 1. 纹理损失
        vgg_gen, vgg_gt = self.vgg(gen), self.vgg(gt)
        l_tex = F.mse_loss(vgg_gen, vgg_gt) * 0.05
        
        # 2. 物理损失 (核心：使用对数差值并大幅缩放，防止 nan 和爆炸)
        gen_gray = torch.mean(gen, dim=1, keepdim=True)
        norm = (gen_gray + 1)/2; sig = torch.sigmoid(15.0*(norm-0.5))
        vf = torch.clamp(torch.mean(sig, [1,2,3]), 1e-4, 0.999)
        # 颗粒计数
        N = torch.sum(F.relu(F.conv2d(norm, self.log, padding='same')), [1,2,3]) + 1.0
        r3 = torch.sum(sig, [1,2,3]) / N
        
        t_safe = torch.clamp(t.view(-1)/100, 1e-3, 10.0)
        # JMAK 误差
        l_j = torch.mean((torch.log(-torch.log(1-vf) + 1e-6) - (self.n*torch.log(t_safe) + self.k))**2)
        # LSW 误差 (对数空间)
        l_l = torch.mean((torch.log(r3 + 1e-3) - torch.log(self.l * t_safe + 1e-3))**2)
        
        l_p = (l_j + l_l) * 0.02 # 归一化到 0.1 左右
        total = Config.LAMBDA_ADV*(-fs.mean()) + Config.LAMBDA_TEX*l_tex + Config.LAMBDA_PHYS*l_p
        return total, l_p, l_tex

# ========================= 5. 训练主循环 (包含实时图片保存) =========================
def start_training():
    os.makedirs(Config.SAMPLE_DIR, exist_ok=True)
    os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)
    print(f"🚀 PIM-GAN V13 (量纲对齐+实时观察) 启动...")
    
    loader = DataLoader(DatasetLoader('train'), batch_size=Config.BATCH_SIZE, shuffle=True)
    G, D = Generator().to(Config.DEVICE), PatchDiscriminator().to(Config.DEVICE)
    loss_fn = CompositeLoss().to(Config.DEVICE)
    
    opt_G = optim.Adam(list(G.parameters())+list(loss_fn.parameters()), lr=Config.LEARNING_RATE, betas=(0.5, 0.999))
    opt_D = optim.Adam(D.parameters(), lr=Config.LEARNING_RATE, betas=(0.5, 0.999))
    scaler = torch.cuda.amp.GradScaler()
    
    for epoch in range(Config.NUM_EPOCHS):
        G.train(); D.train()
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}")
        for b in pbar:
            init, gt, raw_p, norm_p = b['init'].to(Config.DEVICE), b['final'].to(Config.DEVICE), b['raw_p'].to(Config.DEVICE), b['norm_p'].to(Config.DEVICE)
            with torch.backends.cuda.sdp_kernel(enable_flash=False, enable_math=True, enable_mem_efficient=False):
                # Train D
                opt_D.zero_grad()
                with torch.cuda.amp.autocast():
                    gen = G(init, raw_p); rs = D(init, gt, norm_p); fs = D(init, gen.detach(), norm_p)
                    wd = fs.mean() - rs.mean(); alpha = torch.rand(1,1,1,1, device=Config.DEVICE)
                    interp = (alpha*gt + (1-alpha)*gen).requires_grad_(True)
                    grad = torch.autograd.grad(outputs=D(init, interp, norm_p).sum(), inputs=interp, create_graph=True, retain_graph=True)[0]
                    gp = ((grad.view(grad.size(0), -1).norm(2, dim=1)-1)**2).mean()
                    ld = wd + 10*gp
                scaler.scale(ld).backward(); scaler.step(opt_D)

                # Train G
                opt_G.zero_grad()
                with torch.cuda.amp.autocast():
                    gen = G(init, raw_p); fs = D(init, gen, norm_p)
                    lg, lp, lt = loss_fn(gen, gt, raw_p[:,0:1], raw_p[:,1:2], fs)
                
                scaler.scale(lg).backward()
                scaler.unscale_(opt_G)
                torch.nn.utils.clip_grad_norm_(G.parameters(), 1.0) # 🛡️ 强制裁剪防止 nan
                scaler.step(opt_G); scaler.update()

            pbar.set_postfix({'Total': f"{lg.item():.3f}", 'Phys': f"{lp.item():.3f}"})
        
        # --- 实时图片观察 ---
        if (epoch + 1) % 5 == 0:
            G.eval()
            with torch.no_grad():
                def denorm(x): return x * torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).to(Config.DEVICE) + torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).to(Config.DEVICE)
                res = torch.cat([denorm(init[:1]), denorm(gen[:1]), denorm(gt[:1])], dim=3)
                vutils.save_image(res, f"{Config.SAMPLE_DIR}/epoch_{epoch+1}.png", normalize=True)
                print(f"📸 实时图片已保存: epoch_{epoch+1}.png (左:初始 中:预测 右:真实)")
            G.train()

        if (epoch + 1) % Config.SAVE_INTERVAL == 0:
            torch.save(G.state_dict(), f'{Config.CHECKPOINT_DIR}/G_epoch_{epoch+1}.pth')

if __name__ == '__main__':
    DataPreparer().run()
    start_training()